# 05 Basket CDS Pricing

## Credit Risk Basket CDS Pricing

### Objective

The objective of this notebook is to estimate the fair spread of a Basket Credit Default Swap (Basket CDS) using Monte Carlo simulation.

The correlated uniform variables generated by the Gaussian Copula model are transformed into default times. These simulated default times are then used to estimate expected losses for first-to-default through fifth-to-default Basket CDS contracts.

## Step 1 Import Required Libraries

This notebook uses:

- NumPy for numerical calculations
- pandas for data handling
- scipy.stats for probability functions

In [ ]:
## Step 1 Import Required Libraries

This notebook uses:

- NumPy for numerical calculations
- pandas for data handling
- scipy.stats for probability functions

In [1]:
import numpy as np
import pandas as pd

from scipy.stats import norm

## Step 2 Load Model Inputs

The Basket CDS pricing model requires:

- Hazard rates
- Correlation matrix
- Recovery rate

In [2]:
hazard_rates = np.array([
    0.0100,
    0.0125,
    0.0142,
    0.0117,
    0.0133
])

recovery = 0.40

In [4]:
prices = pd.read_csv("../Data/bank_prices.csv", index_col=0)

returns = prices.pct_change().dropna()

corr = returns.corr()

## Step 3 Simulate Correlated Market Scenarios

The Gaussian Copula model is used to generate correlated uniform random variables representing joint default scenarios.

In [5]:
n_sim = 10000

Z = np.random.multivariate_normal(
    np.zeros(5),
    corr,
    n_sim
)

U = norm.cdf(Z)

## Step 4 Estimate Default Times

The simulated uniform variables are transformed into default times using the inverse survival function.

Smaller uniform values correspond to earlier default events.

In [6]:
default_times = -np.log(U) / hazard_rates

default_times[:5]

array([[ 26.2574702 ,  34.29011927,   7.96901954,  11.38763327,
         12.96282956],
       [406.45599242, 202.01972637, 116.95737757, 202.00598682,
        240.48285222],
       [219.38937511,  79.45162843, 167.99489595, 331.16361936,
        230.73763422],
       [ 10.77625047,   8.37887525,  19.00959188,  43.26268843,
         13.45323519],
       [307.43153686, 219.90678812, 113.48705795, 244.38627363,
        124.34735781]])

## Step 5 Rank Default Events

Within each simulation, default times are sorted from earliest to latest.

This allows pricing of first-to-default, second-to-default, and higher-order Basket CDS contracts.

In [7]:
ranked_defaults = np.sort(default_times, axis=1)

ranked_defaults[:5]

array([[  7.96901954,  11.38763327,  12.96282956,  26.2574702 ,
         34.29011927],
       [116.95737757, 202.00598682, 202.01972637, 240.48285222,
        406.45599242],
       [ 79.45162843, 167.99489595, 219.38937511, 230.73763422,
        331.16361936],
       [  8.37887525,  10.77625047,  13.45323519,  19.00959188,
         43.26268843],
       [113.48705795, 124.34735781, 219.90678812, 244.38627363,
        307.43153686]])

## Step 6 Calculate Expected Default Times

Average default times are calculated across all Monte Carlo simulations for each default order.

In [9]:
expected_defaults = ranked_defaults.mean(axis=0)

expected_defaults

array([ 39.74515882,  57.94859737,  76.73215136,  99.44207384,
       136.62883703])

## Step 7 Estimate Basket CDS Spread

A simplified pricing approximation is used in this project.

The Basket CDS spread is estimated from the expected loss:

Spread = Hazard Rate × (1 − Recovery Rate)

This simplified framework demonstrates the pricing workflow before extending to a full premium-leg and protection-leg valuation.

In [10]:
basket_spread = hazard_rates * (1 - recovery)

pricing = pd.DataFrame({
    "Order": [
        "1st-to-Default",
        "2nd-to-Default",
        "3rd-to-Default",
        "4th-to-Default",
        "5th-to-Default"
    ],
    "Expected Default Time": expected_defaults,
    "Approx Spread": basket_spread
})

pricing

,Order,Expected Default Time,Approx Spread
0,1st-to-Default,39.745159,0.00600
1,2nd-to-Default,57.948597,0.00750
2,3rd-to-Default,76.732151,0.00852
3,4th-to-Default,99.442074,0.00702
4,5th-to-Default,136.628837,0.00798


# Business Interpretation

The Basket CDS premium depends on both individual default risk and the dependence structure among the reference entities.

Earlier default events generally result in higher protection values because the protection buyer receives compensation sooner.

The Gaussian Copula model captures the possibility of multiple firms experiencing financial distress during the same market conditions, making correlation a key driver of Basket CDS valuation.

# Summary

In this notebook we:

- Generated correlated market scenarios
- Simulated default times
- Ranked default events
- Estimated expected default times
- Calculated approximate Basket CDS spreads

The pricing results provide the baseline model that will be stress-tested in the sensitivity analysis.